In [31]:
from pathlib import Path
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

import ollama

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [1]:
from pathlib import Path
import pandas as pd

CHUNKS_PATH = Path("../data/processed/annual_report_chunks.csv")

chunks_df = pd.read_csv(CHUNKS_PATH)

print("Number of chunks:", len(chunks_df))
print(chunks_df.head())

Number of chunks: 336
   page  chunk                                               text
0     3      1  PROFILE VISION MISSION MOTTO VALUES PERFORMANC...
1     4      1  PROFILE Â Has been serving Ethiopia since 1942...
2     4      2  e Â We are committed to maintaining the highes...
3     5      1  Iv) Empowerment Â We distinguish employees as ...
4     5      2  ential. Â We are committed to address the need...


In [2]:
!pip install sentence-transformers
!pip install ollama

In [3]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
test_sentence = "The bank reported higher profit."

embedding = model.encode(test_sentence)

print("Embedding type:", type(embedding))
print("Embedding shape:", embedding.shape)

Embedding type: <class 'numpy.ndarray'>
Embedding shape: (384,)


In [5]:
texts = chunks_df["text"].tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Embedding shape: (336, 384)


In [6]:
import numpy as np

EMBEDDINGS_PATH = Path("../data/processed/annual_report_embeddings.npy")

np.save(EMBEDDINGS_PATH, embeddings)

print("Embeddings saved to:", EMBEDDINGS_PATH)

Embeddings saved to: ..\data\processed\annual_report_embeddings.npy


In [7]:
embeddings = np.load(EMBEDDINGS_PATH)

print(embeddings.shape)

(336, 384)


In [8]:
query = "What was the bank's profit?"

query_embedding = model.encode(query)

print(query_embedding.shape)

(384,)


In [9]:
from sklearn.metrics.pairwise import cosine_similarity
scores = cosine_similarity(
    [query_embedding],
    embeddings
)[0]

print(scores.shape)

(336,)


In [10]:
top_k = 5

top_indices = np.argsort(scores)[-top_k:][::-1]

print(top_indices)

[30 21 35 39 41]


In [11]:
for rank, idx in enumerate(top_indices, start=1):
    print("=" * 80)
    print("Rank:", rank)
    print("Page:", chunks_df.iloc[idx]["page"])
    print("Chunk:", chunks_df.iloc[idx]["chunk"])
    print("Score:", scores[idx])
    print("Text:")
    print(chunks_df.iloc[idx]["text"])
    print()

Rank: 1
Page: 21
Chunk: 1
Score: 0.65801215
Text:
Annual Report 2023/24 6 2.1.2 Expense The Bank's total expense in the 2023/24 FY rose by 14% to Birr 110.8 billion. Interest expenses fell by 2.6% unlike in the previous year and comprised 40% of the total. Conversely, non-interest expenses showed a significant growth (29.6%) to reach Birr 66 billion. Profit Before Tax (Mn. Birr) CBE's profitability metrics were sound compared to industry standards. The Bank achieved a Return on Assets (RoA) of 1.9%, a Return on Equity (RoE) of 32.8%, and a Net Interest Margin (NIM) of 4.9%. 2.1.3 Profit During the 2023/24 FY, CBE saw a 20.6% increase in net profit before tax that resulted in Birr 26.7 billion.

Rank: 2
Page: 17
Chunk: 1
Score: 0.52426744
Text:
Annual Report 2023/24 2 to form an opinion as to whether the Bank has complied with sharia Rules and Principles and also with the specific fatwas, rulings and guidelines issued by us and the Bank. The Bank’s management is responsible for ensuring

In [12]:
print("Number of chunks:", len(chunks_df))
print("Embedding shape:", embeddings.shape)

Number of chunks: 336
Embedding shape: (336, 384)


In [17]:
import ollama

In [18]:
response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": "What is a financial annual report?"
        }
    ]
)

print(response["message"]["content"])

A financial annual report, also known as an annual report or AFR, is a detailed financial document that summarizes a company's financial performance over a specific period of time, typically one year. The report provides an overview of the company's financial position, results of operations, and cash flow, as well as a discussion of the company's business strategy and goals.

A typical financial annual report includes the following sections:

1. **Management's Discussion and Analysis (MD&A)**: This section provides an analysis of the company's financial performance, including an explanation of the company's financial position, results of operations, and cash flow. It also discusses the company's business strategy, risks, and future prospects.
2. **Financial Statements**: This section includes the company's balance sheet, income statement, and cash flow statement. The balance sheet provides a snapshot of the company's financial position at a specific point in time, while the income stat

#### Connect Semantic Search + Llama

#### Create the RAG Function

In [19]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import ollama


def ask_financial_assistant(query, top_k=5):

    # ==========================================
    # 1. Convert question into an embedding
    # ==========================================

    query_embedding = model.encode(query)


    # ==========================================
    # 2. Compare question with PDF chunks
    # ==========================================

    scores = cosine_similarity(
        [query_embedding],
        embeddings
    )[0]


    # ==========================================
    # 3. Get the most relevant chunks
    # ==========================================

    top_indices = np.argsort(scores)[-top_k:][::-1]


    # ==========================================
    # 4. Build context for the LLM
    # ==========================================

    context_parts = []

    for rank, idx in enumerate(top_indices, start=1):

        row = chunks_df.iloc[idx]

        context_parts.append(
            f"""
SOURCE {rank}
Page: {row['page']}
Chunk: {row['chunk']}
Similarity Score: {scores[idx]:.4f}

Text:
{row['text']}
"""
        )

    context = "\n".join(context_parts)


    # ==========================================
    # 5. Create the LLM prompt
    # ==========================================

    prompt = f"""
You are an AI Financial Research Assistant.

Answer the user's question using ONLY the
financial report sources provided below.

Rules:

1. Do not invent information.
2. Do not invent financial numbers.
3. If the answer is not found in the sources,
   clearly say that the information was not found.
4. Mention the relevant page number.
5. Keep the answer clear and professional.
6. Use the financial report as the primary source.

FINANCIAL REPORT SOURCES:

{context}


USER QUESTION:

{query}
"""


    # ==========================================
    # 6. Send context to local LLM
    # ==========================================

    response = ollama.chat(
        model="llama3.2:3b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )


    # ==========================================
    # 7. Extract answer
    # ==========================================

    answer = response["message"]["content"]


    # ==========================================
    # 8. Return answer + sources
    # ==========================================

    return {
        "answer": answer,
        "sources": [
            {
                "page": int(chunks_df.iloc[idx]["page"]),
                "chunk": int(chunks_df.iloc[idx]["chunk"]),
                "score": float(scores[idx])
            }
            for idx in top_indices
        ]
    }

#### Test Your Financial Assistant

In [20]:
result = ask_financial_assistant(
    "What was the bank's profit?"
)

print("ANSWER:")
print(result["answer"])

ANSWER:
According to SOURCE 1 (Page 21, Chunk 1), the bank's profit before tax for the 2023/24 FY was Birr 26.7 billion, which represents a 20.6% increase from the previous year.


In [21]:
print("\nSOURCES:")

for source in result["sources"]:
    print(
        f"Page {source['page']} | "
        f"Chunk {source['chunk']} | "
        f"Score {source['score']:.4f}"
    )


SOURCES:
Page 21 | Chunk 1 | Score 0.6580
Page 17 | Chunk 1 | Score 0.5243
Page 24 | Chunk 1 | Score 0.5174
Page 28 | Chunk 1 | Score 0.5157
Page 30 | Chunk 1 | Score 0.5143


In [22]:
ask_financial_assistant(
    "What were the bank's total assets?"
)

{'answer': 'According to SOURCE 3, Page 77, Chunk 1, the total assets of the bank are:\n\nA. Total Assets 150,020 84,734 181,451 208,397 581,963 99,321 1,305,886\n\nAccording to SOURCE 2, Page 118, Chunk 1, the total assets of the bank are:\n\nA. Total Assets 124,351 238,408 162,460 58,198 770,250 80,729 1,434,396',
 'sources': [{'page': 21, 'chunk': 1, 'score': 0.6061792969703674},
  {'page': 118, 'chunk': 1, 'score': 0.5889026522636414},
  {'page': 77, 'chunk': 1, 'score': 0.5800514817237854},
  {'page': 130, 'chunk': 2, 'score': 0.575499415397644},
  {'page': 22, 'chunk': 2, 'score': 0.5571258068084717}]}

In [23]:
ask_financial_assistant(
    "How much did customer deposits increase?"
)

{'answer': 'According to SOURCE 3 (Page 23), customer deposits increased by 11.5% to reach Birr 1,176 billion as of 30 June 2024.\n\nAlso, according to SOURCE 4 (Page 75), customer deposits increased by 11.4% to reach Birr 1,176,035,976,762 as of 30 June 2024.\n\nNo information is available in SOURCE 1 and SOURCE 2 regarding the increase in customer deposits.',
 'sources': [{'page': 117, 'chunk': 2, 'score': 0.5741459131240845},
  {'page': 129, 'chunk': 2, 'score': 0.5629680156707764},
  {'page': 23, 'chunk': 1, 'score': 0.549018144607544},
  {'page': 75, 'chunk': 3, 'score': 0.5458579063415527},
  {'page': 104, 'chunk': 2, 'score': 0.4921278655529022}]}

In [24]:
ask_financial_assistant(
    "How many branches does the bank have?"
)

{'answer': 'According to SOURCE 1, Page 23, Chunk 2, the Bank had a total of 1,942 branches as of 30 June 2024.',
 'sources': [{'page': 23, 'chunk': 2, 'score': 0.566628634929657},
  {'page': 24, 'chunk': 1, 'score': 0.5363794565200806},
  {'page': 28, 'chunk': 1, 'score': 0.5277661085128784},
  {'page': 30, 'chunk': 1, 'score': 0.5272191762924194},
  {'page': 26, 'chunk': 1, 'score': 0.525928258895874}]}

### Step 10 — Improve the Retrieval System

#### Create a Search Function

In [26]:
def semantic_search(query, top_k=5):

    # Convert question to embedding
    query_embedding = model.encode(query)

    # Calculate similarity
    scores = cosine_similarity(
        [query_embedding],
        embeddings
    )[0]

    # Get highest scores
    top_indices = np.argsort(scores)[-top_k:][::-1]

    results = []

    for rank, idx in enumerate(top_indices, start=1):

        row = chunks_df.iloc[idx]

        results.append({
            "rank": rank,
            "page": int(row["page"]),
            "chunk": int(row["chunk"]),
            "score": float(scores[idx]),
            "text": row["text"]
        })

    return results

In [27]:
results = semantic_search(
    "How many branches does the bank have?",
    top_k=5
)

In [28]:
for result in results:

    print("=" * 80)

    print("Rank:", result["rank"])
    print("Page:", result["page"])
    print("Chunk:", result["chunk"])
    print("Score:", round(result["score"], 4))

    print("\nText:")
    print(result["text"])

Rank: 1
Page: 23
Chunk: 2
Score: 0.5666

Text:
 of 7.3 million. Likewise, the number of digital channel users saw significant growth, with debit card holders, mobile banking users, and CBE Birr users becoming 31.4 million, 10.7 million and 34.2 million, respectively. 3.2 Accessibility In the 2023/24 FY, the Bank opened five new branches, increasing the total number of branches to 1,942 as of 30 June 2024. Among these, 155 offer only CBE NOOR services, and 1,785 offer these services in separate windows. Reflecting the Bank’s emphasis on digitalization, the number of new branches opened fell significantly from the previous year’s total of 113. To enhance the accessibility of services via digital channels, CBE has been actively recruiting agents and merchants for CBE Birr. As of 30 June 2024, the Bank had engaged 74 super agents, 42,145 agents and 58,125 merchants. 3.3 Human Resource Development By the end of June 2024, the total workforce of the Bank reached 81,980, consisting of 48,729 

#### Add a Similarity Threshold

In [29]:
def semantic_search(query, top_k=5, min_score=0.35):

    query_embedding = model.encode(query)

    scores = cosine_similarity(
        [query_embedding],
        embeddings
    )[0]

    top_indices = np.argsort(scores)[-top_k:][::-1]

    results = []

    for rank, idx in enumerate(top_indices, start=1):

        score = float(scores[idx])

        if score < min_score:
            continue

        row = chunks_df.iloc[idx]

        results.append({
            "rank": rank,
            "page": int(row["page"]),
            "chunk": int(row["chunk"]),
            "score": score,
            "text": row["text"]
        })

    return results

In [30]:
results = semantic_search(
    "How many branches does the bank have?",
    top_k=5,
    min_score=0.35
)